## Generate Embeddings


In [1]:
import pandas as pd

dataset = pd.read_csv("embedded_sentences.csv")
dataset.head(10)

,sentence
0,A little girl is smiling and running outside
1,A man is drawing on a digital dry erase board
2,A black bird is sitting on a dead tree
3,An elderly man is sitting on a bench
4,A man and a woman are sitting comfortably on t...
5,A sheepdog is grouping a herd of sheep
6,A man is opening a small package that contains...
7,The man in a t-shirt dyed grey is sitting on a...
8,A woman is not slicing a cucumber


In [1]:
## this is for gemnin

import os

import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

EMBEDDING_DIM = 768  # choose 768, 1536 or 3072


def get_embedding(sentence, task_type="SEMANTIC_SIMILARITY"):
    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=sentence,
        config=types.EmbedContentConfig(
            task_type=task_type,
            output_dimensionality=EMBEDDING_DIM,
        ),
    )
    vec = np.array(result.embeddings[0].values)
    return (vec / np.linalg.norm(vec)).tolist()  # normalize

In [3]:
import os
import numpy as np
import pandas as pd

if os.path.exists("embedded_sentences.csv"):
    dataset = pd.read_csv("embedded_sentences.csv")
    dataset["embedding"] = dataset.embedding.apply(eval)
else:
    dataset["embedding"] = dataset["sentence"].apply(get_embedding)
    dataset.to_csv("embedded_sentences.csv", index=False)


In [4]:
dataset["id"] = range(1, len(dataset) + 1)
dataset.head()

,sentence,embedding,id
0,A little girl is smiling and running outside,"[-0.027314, 0.024023, -0.000741, -0.086041, -0...",1
1,A man is drawing on a digital dry erase board,"[-0.042826, 0.008481, 0.030796, -0.102968, -0....",2
2,A black bird is sitting on a dead tree,"[-0.002864, 0.014019, 0.010561, -0.160566, -0....",3
3,An elderly man is sitting on a bench,"[-0.047324, -0.002194, -0.008719, -0.135576, -...",4
4,A man and a woman are sitting comfortably on t...,"[-0.040565, -0.004491, -0.029225, -0.123033, -...",5


In [5]:
embedding_dimension = len(dataset.iloc[0]["embedding"])
embedding_dimension

768

In [6]:
dataset['embedding'][0]

[-0.02731440769061451,
 0.024023310557737574,
 -0.0007406451834097946,
 -0.08604090789911195,
 -0.005834451316264451,
 0.013171032127197462,
 -0.03486287242299178,
 0.026035646610119784,
 0.01754770170645397,
 0.012428259793652642,
 0.04148154391893581,
 -0.007330480423866866,
 -0.021134148393701483,
 0.05445098567106719,
 0.239737860390301,
 0.02322708838058292,
 -0.003193222934049778,
 0.024821902632451917,
 0.023271772002250547,
 -0.03071413626665748,
 -0.004614502422426218,
 -0.010935524723706756,
 0.0016399257574271923,
 -0.0030630725468671076,
 -0.016401349869948262,
 -0.0201105548517588,
 0.03429884537174627,
 -0.010389418416263977,
 0.02192621682415717,
 0.0011315943832996332,
 -0.004510639905441382,
 0.011515051556157514,
 0.023997450742789493,
 -0.042247444087906365,
 -0.006161166970347058,
 -0.01747080769835873,
 0.06581749417303762,
 0.03006356425789074,
 0.03739174161428095,
 0.02672079718928195,
 -0.04830703909952457,
 -0.03604377712601482,
 0.0036488034386922973,
 -0.028

In [7]:
embeddings = np.array(dataset.embedding.tolist())

query = "I love soccer"
xq = get_embedding(query)

In [8]:
xq

[-0.0034452482279799922,
 0.011365492491404058,
 0.018777735968417898,
 -0.12166101543439739,
 -0.05937419152954507,
 0.048525747454869,
 0.01319895550712965,
 -0.002090941373957385,
 0.026493277680751787,
 0.017328222982419286,
 0.0022860522422352677,
 -0.04302188773968902,
 -0.05096342505701343,
 0.04438796350469139,
 0.19840253859823045,
 -0.026371519207629424,
 -0.018915473225971202,
 -0.009634750523645765,
 -0.0008746975980712101,
 0.011211854510601187,
 -0.044567432948600445,
 0.003950913966877795,
 0.0021153670570059424,
 -0.01614206975557438,
 -0.021038604994748834,
 0.009978363965181392,
 0.05499159922966565,
 0.03329673048913602,
 0.03759898595450553,
 0.009156405689134414,
 -0.023497997181173045,
 0.03698190383082819,
 0.021514167287540617,
 0.033769070212722796,
 0.015390953189512406,
 0.02577746682943979,
 0.051139178109896176,
 0.05860575216914675,
 0.033630143370644634,
 0.051936873443487935,
 -0.04024948379181023,
 -0.007232918785686769,
 0.018269927619789583,
 -0.01644

In [9]:
def search(query, k=5):
    xq = np.array(get_embedding(query))
    scores = embeddings @ xq
    top_idx = np.argsort(scores)[::-1][:k]
    return dataset.iloc[top_idx].assign(score=scores[top_idx])[["id", "sentence", "score"]]


search("a man id working hard",5)

,id,sentence,score
1,2,A man is drawing on a digital dry erase board,0.803530
6,7,A man is opening a small package that contains...,0.791304
7,8,The man in a t-shirt dyed grey is sitting on a...,0.783492
3,4,An elderly man is sitting on a bench,0.769594
5,6,A sheepdog is grouping a herd of sheep,0.743210


## Pinecone


In [10]:
import os

from pinecone import Pinecone

PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")
database = Pinecone(api_key=PINECONE_API_KEY)

In [11]:
# hERE WE ARE CREATING AN INDEX ON PINECONE
from pinecone import ServerlessSpec

serverless_spec = ServerlessSpec(cloud="aws", region="us-east-1")

In [12]:


import time

INDEX_NAME = "banoqabil-fm"

# delete the old schema index if it exists
if INDEX_NAME in database.list_indexes().names():
    database.delete_index(INDEX_NAME)
    time.sleep(5)

database.create_index(
    name=INDEX_NAME,
    dimension=768,            # or embedding_dimension, if that variable is 768
    metric="cosine",
    spec=serverless_spec,
)

# wait until the index is ready
while not database.describe_index(INDEX_NAME).status["ready"]:
    time.sleep(1)

pinecone_index = database.Index(INDEX_NAME)
print(database.describe_index(INDEX_NAME))

IndexModel(name='banoqabil-fm', status='Ready', host='https://banoqabil-fm-fsu6fnd.svc.aped-4627-b74a.pinecone.io', deployment='Managed', deletion_protection='disabled', schema_fields=2)


In [13]:
pinecone_index.describe_index_stats()

DescribeIndexStatsResponse(dimension=768, total_vector_count=0, metric='cosine', namespaces=0)

In [14]:
def iterator(dataset, size):
    for i in range(0, len(dataset), size):
        yield dataset.iloc[i : i + size]


def vector(batch):
    vector = []
    for i in batch.to_dict("records"):
        vector.append((str(i["id"]), i["embedding"], {"sentence": i["sentence"]}))

    return vector

In [15]:
# if pinecone_index.describe_index_stats()["total_vector_count"] == 0:
#     pinecone_index.upsert(vectors=vector(dataset))

pinecone_index.upsert(vectors=vector(dataset))

UpsertResponse(upserted_count=9)

In [16]:
pinecone_index.describe_index_stats()

DescribeIndexStatsResponse(dimension=768, total_vector_count=9, metric='cosine', namespaces=1)

In [17]:
response = pinecone_index.query(vector=xq, top_k=4, include_metadata=True)
for match in response["matches"]:
    print(match["metadata"]["sentence"])

A little girl is smiling and running outside
A sheepdog is grouping a herd of sheep
A man and a woman are sitting comfortably on the bench
The man in a t-shirt dyed grey is sitting on a rock in front of a waterfall


In [18]:
query2 = "a sparrow is eating"
xq2 = get_embedding(query2)
response = pinecone_index.query(vector=xq2, top_k=5, include_metadata=True)
for match in response["matches"]:
    print(match["metadata"]["sentence"])

A black bird is sitting on a dead tree
A woman is not slicing a cucumber
A man is opening a small package that contains headphones
The man in a t-shirt dyed grey is sitting on a rock in front of a waterfall
An elderly man is sitting on a bench


In [19]:
query2 = "a sparrow is eating"
xq2 = get_embedding(query2)
response = pinecone_index.query(vector=xq2, top_k=5, include_metadata=True)

for match in response["matches"]:
    print(f"{match['score']:.4f}  {match['metadata']['sentence']}")

0.7974  A black bird is sitting on a dead tree
0.7150  A woman is not slicing a cucumber
0.7101  A man is opening a small package that contains headphones
0.7068  The man in a t-shirt dyed grey is sitting on a rock in front of a waterfall
0.7057  An elderly man is sitting on a bench
